In [10]:
# === Core Python ===
import os
import glob
import re
import collections
import cftime
from datetime import datetime
from typing import Tuple, Dict, Optional, List
from pathlib import Path
from typing import Any, Dict, Mapping, Optional, Sequence, Tuple

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs
from xskillscore import rmse, pearson_r

from dataclasses import dataclass

from DataUtil import (
    build_experiments,
    DEFAULT_EXPERIMENTS
)

from ObsUtil import (
    OBS_REGISTRY,
    get_obs_file,
    list_obs_sources,
    obs_coverage,
)

In [14]:
@dataclass(frozen=True)
class FileCollectionConfig:
    group: str
    freq: str
    run: str
    obs: str
    period: str
    model_template: str            # <-- REQUIRED, explicit
    ens_prefix: str = "en"
    ens_width: int = 2
    ens_start: int = 1 

    def parse_period(self) -> Tuple[int, int, int, int]:
        m = re.match(r"^(\d{4})(\d{2})-(\d{4})(\d{2})$", self.period)
        if not m:
            raise ValueError(f"Bad period '{self.period}', expected 'YYYYMM-YYYYMM'")
        y0, m0, y1, m1 = map(int, m.groups())
        if (y1, m1) < (y0, m0):
            raise ValueError(f"Bad period '{self.period}' (end < start)")
        return y0, m0, y1, m1

    def years(self) -> List[int]:
        y0, _, y1, _ = self.parse_period()
        return list(range(y0, y1 + 1))

    def ens_labels(self, nens: int) -> List[str]:
        return [
            f"{self.ens_prefix}{i:0{self.ens_width}d}"
            for i in range(self.ens_start, nens + self.ens_start)
        ]
    
class S2SFileCollector:
    """
    Collect obs + model files for S2S / ACC diagnostics.
    """

    def __init__(
        self,
        *,
        exp_list: dict,
        exp_dict: dict,
        obs_registry,
        s2s_var_dict: Dict[str, str],
        get_obs_file_func,
    ):
        self.exp_list = exp_list
        self.exp_dict = exp_dict
        self.obs_registry = obs_registry
        self.s2s_var_dict = s2s_var_dict
        self.get_obs_file = get_obs_file_func

    # ---------- path resolvers ----------

    def resolve_obs_file(
        self,
        obs: str,
        freq: str,
        year: int,
        var: Optional[str] = None,
    ) -> str:
        return self.get_obs_file(
            self.obs_registry,
            obs,
            freq=freq,
            year=year,
            var=var,
        )
        
    def resolve_model_file(
        self,
        *,
        run_meta,
        freq: str,
        year: int,
        var: str,
        ens: str,
        template: str,
    ) -> str:
        ts_dir = self.model_ts_dir(run_meta, freq)
        fname = template % {
            "var": var,
            "ens": ens,
            "year": year,
            "period": run_meta.period,
        }
        return os.path.join(ts_dir, fname)

    # ---------- model file discovery ----------

    def candidates_model_files(
        self,
        *,
        ts_dir: str,
        var: str,
        ens: str,
        years: List[int],
        period: str,
        templates: List[str],
    ) -> List[str]:
        out: List[str] = []
        for tpl in templates:
            for y in years:
                name = tpl % {
                    "var": var,
                    "ens": ens,
                    "year": y,
                    "period": period,
                }
                path = os.path.join(ts_dir, name)
                if os.path.exists(path):
                    out.append(path)

        # de-duplicate while preserving order
        seen = set()
        uniq = []
        for p in out:
            if p not in seen:
                uniq.append(p)
                seen.add(p)
        return uniq

    def model_ts_dir(self, run_meta, freq: str) -> str:
        return os.path.join(run_meta.atm_path, "ts", freq)

    def model_files_for_ens(
        self,
        cfg: FileCollectionConfig,
        *,
        run_meta,
        var: str,
        ens: str,
    ) -> List[str]:
        ts_dir = self.model_ts_dir(run_meta, cfg.freq)
        years = cfg.years()
        return [
            os.path.join(
                ts_dir,
                cfg.model_template % {"var": var, "ens": ens, "year": y, "period": cfg.period},
            )
            for y in years
        ]
        
    # ---------- main collection ----------
    def collect_one_var(
        self,
        cfg: FileCollectionConfig,
        *,
        var: str,
        obs_var: str,
        verbose: bool = True
    ):
        years = cfg.years()
        models = self.exp_list[cfg.group]["models"]

        # --- obs files ---
        obs_files = [self.resolve_obs_file(cfg.obs, cfg.freq, y, obs_var) for y in years]
        missing_obs = [f for f in obs_files if not os.path.exists(f)]
        if missing_obs:
            if verbose:
                print(f"[MISSING OBS] {var} (obs_var={obs_var})")
                for f in missing_obs:
                    print("  ", f)
            return None

        out_var = {}

        for exp in models:
            run_meta = self.exp_dict[exp]["runs"][cfg.run]
            if run_meta is None:
                if verbose:
                    print(f"SKIP {exp}: no run='{cfg.run}'")
                continue

            nens = int(self.exp_dict[exp]["nens"])
            ts_dir = self.model_ts_dir(run_meta, cfg.freq)

            model_by_ens = {}
            missing_any = False

            for ens in cfg.ens_labels(nens):
                files = self.model_files_for_ens(cfg, run_meta=run_meta, var=var, ens=ens)

                missing = [f for f in files if not os.path.exists(f)]
                if missing:
                    missing_any = True
                    if verbose:
                        print(f"[MISSING MOD] {exp} {var} {ens} (ts_dir={ts_dir})")
                        for f in missing[:5]:
                            print("   ", f)
                        if len(missing) > 5:
                            print(f"   ... {len(missing)-5} more")

                model_by_ens[ens] = files  # keep full list for later loading

            out_var[exp] = {
                "obs": obs_files,
                "model": model_by_ens,
                "ts_dir": ts_dir,
                "nens": nens,
            }

            if verbose and missing_any:
                print(f"[WARN] {exp} {var}: some ensemble members missing files")

        return out_var
        
    def collect(
        self,
        cfg: FileCollectionConfig,
        *,
        vars_to_process: Optional[List[str]] = None,
        verbose: bool = True,
    ) -> Dict[str, dict]:

        out = {}

        for var, obs_var in self.s2s_var_dict.items():
            if vars_to_process is not None and var not in vars_to_process:
                continue

            res = self.collect_one_var(cfg, var=var, obs_var=obs_var, verbose=verbose)
            if res is None:
                continue
            out[var] = res

        return out

@dataclass(frozen=True)
class BiweeklyAccMapConfig:
    lat_range: Optional[Tuple[float, float]] = (-20.0, 20.0)
    lon_range: Optional[Tuple[float, float]] = None

    init_time: str = "2012-01-01"
    lead_bins: Tuple[Tuple[int, int], ...] = ((1, 14), (15, 28), (29, 42), (43, 56))
    lead_bin_labels: Tuple[str, ...] = ("wk1-2", "wk3-4", "wk5-6", "wk7-8")
    min_samples: int = 15

    require_same_grid: bool = True
    grid_tol: float = 0.0
    require_nonempty_time: bool = True

    # new
    compute_member_stats: bool = True
    save_member_maps: bool = False   # WARNING: can be large

    out_nc: str = "s2s_acc_map_biweekly.nc"
    overwrite: bool = True
    verbose : bool = True 

class BiweeklyAccMapAssessor:
    """
    Biweekly ACC-map assessor.

    Output variables (always):
      - ACC_EMEAN(var, exp, lead_bin, lat, lon)
      - N_SAMPLES_EMEAN(var, exp, lead_bin, lat, lon)

    If cfg.compute_member_stats:
      - ACC_MEMBER_MEAN(var, exp, lead_bin, lat, lon)
      - ACC_MEMBER_STD(var, exp, lead_bin, lat, lon)
      - P_ACC_POS(var, exp, lead_bin, lat, lon)      # fraction of members with ACC>0
      - N_ENS_USED(var, exp)                         # count of members included (non-missing)
      - N_SAMPLES_MEMBER_MEAN(var, exp, lead_bin, lat, lon)  # mean paired samples across members

    If cfg.save_member_maps:
      - ACC_MEMBER(var, exp, ens, lead_bin, lat, lon)
      - N_SAMPLES_MEMBER(var, exp, ens, lead_bin, lat, lon)
    """

    def __init__(self, cfg: BiweeklyAccMapConfig):
        self.cfg = cfg
        self._validate_cfg()

    # ----------------------------
    # Helpers (mostly unchanged)
    # ----------------------------
    def _validate_cfg(self) -> None:
        if len(self.cfg.lead_bins) != len(self.cfg.lead_bin_labels):
            raise ValueError("lead_bins and lead_bin_labels must have same length.")
        if self.cfg.min_samples < 2:
            raise ValueError("min_samples must be >= 2.")
        for a, b in self.cfg.lead_bins:
            if a > b:
                raise ValueError(f"Invalid lead bin ({a},{b}).")
        _ = np.datetime64(self.cfg.init_time)

    @staticmethod
    def _parse_yyyymm_period(period: str) -> Tuple[int, int, int, int]:
        m = re.match(r"^(\d{4})(\d{2})-(\d{4})(\d{2})$", period)
        if not m:
            raise ValueError(f"Bad period '{period}', expected 'YYYYMM-YYYYMM'")
        y0, m0, y1, m1 = map(int, m.groups())
        if (y1, m1) < (y0, m0):
            raise ValueError(f"Bad period '{period}' (end < start)")
        return y0, m0, y1, m1

    @classmethod
    def _select_period_time(cls, da: xr.DataArray, period: str) -> xr.DataArray:
        y0, m0, y1, m1 = cls._parse_yyyymm_period(period)
        t0 = np.datetime64(f"{y0:04d}-{m0:02d}-01")
        if m1 == 12:
            t1 = np.datetime64(f"{y1 + 1:04d}-01-01")
        else:
            t1 = np.datetime64(f"{y1:04d}-{m1 + 1:02d}-01")
        return da.sel(time=slice(t0, t1))

    @staticmethod
    def _open_var(files: List[str], var: str) -> xr.DataArray:
        ds = xr.open_mfdataset(files, combine="by_coords")
        if var in ds:
            return ds[var]
        for k in ds.data_vars:
            if k.lower() == var.lower():
                return ds[k]
        raise KeyError(f"Variable '{var}' not found. Available={list(ds.data_vars)}")

    @staticmethod
    def _guess_lat_lon_names(da: xr.DataArray) -> Tuple[str, str]:
        lat_candidates = ["lat", "latitude", "LAT", "nlat"]
        lon_candidates = ["lon", "longitude", "LON", "nlon"]
        lat = next((n for n in lat_candidates if (n in da.dims or n in da.coords)), None)
        lon = next((n for n in lon_candidates if (n in da.dims or n in da.coords)), None)
        if lat is None or lon is None:
            raise ValueError(f"Cannot infer lat/lon from dims={da.dims}, coords={list(da.coords)}")
        return lat, lon

    @staticmethod
    def _subset_region(
        da: xr.DataArray,
        lat_name: str,
        lon_name: str,
        lat_range: Optional[Tuple[float, float]],
        lon_range: Optional[Tuple[float, float]],
    ) -> xr.DataArray:
        out = da
        if lat_range is not None:
            lo, hi = lat_range
            latv = out[lat_name]
            if latv.size > 1 and (latv[0] > latv[-1]):
                out = out.sel({lat_name: slice(hi, lo)})
            else:
                out = out.sel({lat_name: slice(lo, hi)})
        if lon_range is not None:
            lo, hi = lon_range
            out = out.sel({lon_name: slice(lo, hi)})
        return out

    @staticmethod
    def _same_1d_values(a: xr.DataArray, b: xr.DataArray, *, tol: float = 0.0) -> bool:
        if a.size != b.size:
            return False
        av = np.asarray(a.values)
        bv = np.asarray(b.values)
        if tol == 0.0:
            return np.array_equal(av, bv)
        return np.allclose(av, bv, rtol=0.0, atol=tol, equal_nan=True)

    @staticmethod
    def _rename_dim_if_present(da: xr.DataArray, old: str, new: str) -> xr.DataArray:
        if old == new:
            return da
        rename_map = {}
        if old in da.dims:
            rename_map[old] = new
        if old in da.coords:
            rename_map[old] = new
        if rename_map:
            return da.rename(rename_map)
        return da

    @classmethod
    def _ensure_latlon_names_match(
        cls,
        mod: xr.DataArray,
        obs: xr.DataArray,
        *,
        mod_lat: str,
        mod_lon: str,
        obs_lat: str,
        obs_lon: str,
        tol: float = 0.0,
    ) -> xr.DataArray:
        if (mod_lat == obs_lat) and (mod_lon == obs_lon):
            return mod

        if mod_lat not in mod.coords or mod_lon not in mod.coords:
            raise ValueError(f"Model missing coords '{mod_lat}'/'{mod_lon}'. coords={list(mod.coords)}")
        if obs_lat not in obs.coords or obs_lon not in obs.coords:
            raise ValueError(f"Obs missing coords '{obs_lat}'/'{obs_lon}'. coords={list(obs.coords)}")

        lat_ok = cls._same_1d_values(mod[mod_lat], obs[obs_lat], tol=tol)
        lon_ok = cls._same_1d_values(mod[mod_lon], obs[obs_lon], tol=tol)

        if not (lat_ok and lon_ok):
            raise ValueError("Model/obs lat/lon values differ; regrid explicitly.")

        mod = cls._rename_dim_if_present(mod, mod_lat, obs_lat)
        mod = cls._rename_dim_if_present(mod, mod_lon, obs_lon)
        return mod

    @staticmethod
    def _to_daily_date_time(obj: xr.DataArray | xr.Dataset) -> xr.DataArray | xr.Dataset:
        if "time" not in obj.coords:
            raise KeyError("Object has no 'time' coordinate.")
        t = obj["time"]
        if np.issubdtype(t.dtype, np.datetime64):
            t_daily = t.astype("datetime64[D]")
        else:
            t_daily = xr.DataArray(
                np.array([np.datetime64(str(v)[:10]) for v in t.values], dtype="datetime64[D]"),
                dims=t.dims,
                coords=t.coords,
                name=t.name,
            )
        return obj.assign_coords(time=t_daily)

    def _align_and_check(self, mod: xr.DataArray, obs: xr.DataArray, *, lat_name: str, lon_name: str):
        mod, obs = xr.align(mod, obs, join="inner")
        mod = mod.sortby("time")
        obs = obs.sortby("time")
        if self.cfg.require_nonempty_time and mod.sizes.get("time", 0) == 0:
            raise ValueError("No overlapping times between model and obs.")
        if self.cfg.require_same_grid:
            if not self._same_1d_values(mod[lat_name], obs[lat_name], tol=self.cfg.grid_tol):
                raise ValueError("lat grids differ; regrid or disable require_same_grid.")
            if not self._same_1d_values(mod[lon_name], obs[lon_name], tol=self.cfg.grid_tol):
                raise ValueError("lon grids differ; regrid or disable require_same_grid.")
        return mod, obs

    @classmethod
    def _concat_ens_mean(cls, model_files_by_ens: Dict[str, List[str]], *, var: str, period: str) -> xr.DataArray:
        members = []
        ens_names = []
        for ens, files in model_files_by_ens.items():
            if not files:
                continue
            da = cls._open_var(files, var)
            da = cls._select_period_time(da, period)
            members.append(da)
            ens_names.append(ens)
        if not members:
            raise ValueError(f"No model files found for var={var} across ensembles.")
        ens_da = xr.concat(members, dim=xr.IndexVariable("ens", ens_names))
        return ens_da.mean("ens")

    # ----------------------------
    # NEW: ACC-map core
    # ----------------------------
    def _add_lead_day_coord(self, da: xr.DataArray) -> xr.DataArray:
        init = np.datetime64(self.cfg.init_time)
        t = da["time"].values
        lead = ((t - init) / np.timedelta64(1, "D")).astype("float32")
        return da.assign_coords(lead_day=("time", lead))

    @staticmethod
    def _maybe_convert_geopotential_to_height(
        da: xr.DataArray,
        *,
        var_key: str,
        prefer: str = "m",   # normalize to meters
        g: float = 9.80665,
        verbose: bool = False,
    ) -> xr.DataArray:
        """
        Ensure geopotential-type variables are in geopotential height (m).

        Handles:
          - ERA-style geopotential: units like "m2 s-2", "m^2/s^2", "m**2 s**-2"
          - Height already: units like "m", "meter"
          - Heuristic fallback based on magnitude if units missing/garbled

        var_key: your loop variable name (e.g., "Z500", "Z850", "Z", etc.)
        """

        # Only attempt for Z-like variables.
        # Covers: Z, Zxxx, Z500, Z850, zg, geopotential, etc.
        k = (var_key or "").lower()
        name = (da.name or "").lower()
        if not (
            k.startswith("z") or
            "geopot" in k or "geopot" in name or
            name.startswith("z") or name in ("zg", "phi")
        ):
            return da

        units = (da.attrs.get("units") or "").strip().lower().replace("^", "").replace("**", "")
        units = units.replace(" ", "")

        def _is_geopotential_units(u: str) -> bool:
            # common encodings of m^2 s^-2
            return (
                "m2s-2" in u or
                "m2/s2" in u or
                "m2s**-2" in u or
                "m2s^-2" in u
            )

        def _is_height_units(u: str) -> bool:
            return u in ("m", "meter", "metre", "meters", "metres")

        # If units tell us clearly
        if units:
            if _is_geopotential_units(units):
                if verbose:
                    print(f"[UNIT FIX] {da.name or var_key}: geopotential -> height (divide by g)")
                out = da / g
                out.attrs = dict(da.attrs)
                out.attrs["units"] = "m"
                out.attrs["note_unit_fix"] = "converted from geopotential (m2 s-2) via /g"
                return out

            if _is_height_units(units):
                return da

        # Fallback heuristic (units missing or untrusted):
        # geopotential ~ O(1e4-1e5); height ~ O(1e3-1e4)
        try:
            vmax = float(da.max(skipna=True))
        except Exception:
            return da

        # These thresholds are conservative and work across pressure levels.
        # Heights at low levels can be small, but won't exceed ~2e4 m.
        if vmax > 2.0e4 and vmax < 5.0e6:
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: heuristic geopotential -> height (vmax={vmax:.3g})")
            out = da / g
            out.attrs = dict(da.attrs)
            out.attrs["units"] = "m"
            out.attrs["note_unit_fix"] = "heuristic: treated as geopotential and converted via /g"
            return out

        return da
        
    @staticmethod
    def _maybe_convert_precip_to_mmday(
        da: xr.DataArray,
        *,
        var_key: str,
        verbose: bool = False,
    ) -> xr.DataArray:
        """
        Normalize precipitation-like variables to mm/day.

        Handles common units:
          - m/s
          - kg m-2 s-1  (or kg/m^2/s)
          - mm/day, mm d-1
          - mm/s, m/day, etc.
        Falls back to a magnitude heuristic if units missing.
        """
        k = (var_key or "").lower()
        name = (da.name or "").lower()

        # decide if this looks like precipitation
        is_pr = (
            k in ("pr", "precip", "prect", "precc", "precl", "tp") or
            "precip" in k or "precip" in name or
            k.startswith("pr") or name.startswith("pr") or
            name in ("prect", "precc", "precl", "tp")
        )
        if not is_pr:
            return da

        u = (da.attrs.get("units") or "").strip().lower()
        u0 = u.replace(" ", "")

        def _set_units(out: xr.DataArray, note: str) -> xr.DataArray:
            out.attrs = dict(da.attrs)
            out.attrs["units"] = "mm/day"
            out.attrs["note_unit_fix"] = note
            return out

        # already mm/day-ish
        if u0 in ("mm/day", "mmd-1", "mm/d", "mmdy-1") or ("mm" in u0 and ("day" in u0 or "d-1" in u0)):
            return da

        # kg m-2 s-1  -> mm/day
        if ("kg" in u0 and "m-2" in u0 and "s-1" in u0) or u0 in ("kgm-2s-1", "kg/m2/s", "kgm**-2s**-1"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: kg m-2 s-1 -> mm/day (×86400)")
            out = da * 86400.0
            return _set_units(out, "converted from kg m-2 s-1 via ×86400")

        # m/s -> mm/day
        if u0 in ("m/s", "ms-1", "msec-1") or ("m" in u0 and "s-1" in u0 and "kg" not in u0 and "mm" not in u0):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: m/s -> mm/day (×86400×1000)")
            out = da * 86400.0 * 1000.0
            return _set_units(out, "converted from m s-1 via ×86400×1000")

        # mm/s -> mm/day
        if u0 in ("mm/s", "mms-1"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: mm/s -> mm/day (×86400)")
            out = da * 86400.0
            return _set_units(out, "converted from mm s-1 via ×86400")

        # m/day -> mm/day
        if u0 in ("m/day", "md-1", "m/d"):
            if verbose:
                print(f"[UNIT FIX] {da.name or var_key}: m/day -> mm/day (×1000)")
            out = da * 1000.0
            return _set_units(out, "converted from m day-1 via ×1000")

        # Fallback heuristic (if units missing): typical precip rates are small in SI.
        # If values are ~1e-7 to 1e-4 it might be m/s; if ~1e-6 to 1e-3 might be kg/m2/s.
        # We DO NOT auto-convert without some confidence; use a conservative rule.
        if not u0:
            try:
                vmax = float(da.max(skipna=True))
            except Exception:
                return da

            # If vmax is tiny (<1e-2) it's likely a rate in per-second units (m/s or kg/m2/s).
            # Distinguish by checking if it's closer to m/s scale (very tiny) vs kg/m2/s (slightly larger),
            # but both can overlap. We'll choose kg/m2/s unless extremely tiny.
            if 0 < vmax < 1e-2:
                # extremely tiny -> treat as m/s
                if vmax < 5e-5:
                    if verbose:
                        print(f"[UNIT FIX] {da.name or var_key}: heuristic m/s -> mm/day (vmax={vmax:.3g})")
                    out = da * 86400.0 * 1000.0
                    return _set_units(out, "heuristic: treated as m s-1 and converted to mm/day")
                else:
                    if verbose:
                        print(f"[UNIT FIX] {da.name or var_key}: heuristic kg m-2 s-1 -> mm/day (vmax={vmax:.3g})")
                    out = da * 86400.0
                    return _set_units(out, "heuristic: treated as kg m-2 s-1 and converted to mm/day")

        return da

    @staticmethod
    def _nan_corr_1d(x: np.ndarray, y: np.ndarray) -> float:
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 2:
            return np.nan
        x0 = x[m] - np.mean(x[m])
        y0 = y[m] - np.mean(y[m])
        den = np.sqrt(np.sum(x0 * x0) * np.sum(y0 * y0))
        if den == 0:
            return np.nan
        return float(np.sum(x0 * y0) / den)

    def _select_lead_bin(self, da: xr.DataArray, a: int, b: int) -> xr.DataArray:
        m = (da["lead_day"] >= float(a)) & (da["lead_day"] <= float(b))
        return da.sel(time=da.time.where(m, drop=True))

    def _corr_map_time(self, fc: xr.DataArray, obs: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
        """
        Correlation across time at each (lat,lon). Returns (acc_map, n_samples_map).
        Handles dask chunking on time core dim.
        """
        tname = "time"

        # Fix for your error: core dim time must be single chunk for dask='parallelized'
        if hasattr(fc.data, "chunks"):
            fc = fc.chunk({tname: -1})
        if hasattr(obs.data, "chunks"):
            obs = obs.chunk({tname: -1})

        paired_valid = xr.apply_ufunc(
            lambda a, b: np.isfinite(a) & np.isfinite(b),
            fc, obs,
            input_core_dims=[[tname], [tname]],
            output_core_dims=[[tname]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[bool],
        )
        n_samples = paired_valid.sum(tname).astype("int32")

        acc = xr.apply_ufunc(
            self._nan_corr_1d,
            fc, obs,
            input_core_dims=[[tname], [tname]],
            output_core_dims=[[]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float],
        ).astype("float32")

        acc = acc.where(n_samples >= self.cfg.min_samples)
        return acc, n_samples

    def _compute_biweekly_maps(self, fc: xr.DataArray, obs: xr.DataArray) -> tuple[xr.DataArray, xr.DataArray]:
        acc_list = []
        ns_list = []
        for (a, b), lab in zip(self.cfg.lead_bins, self.cfg.lead_bin_labels):
            fc_bin = self._select_lead_bin(fc, a, b)
            obs_bin = self._select_lead_bin(obs, a, b)

            if fc_bin.sizes.get("time", 0) == 0 or obs_bin.sizes.get("time", 0) == 0:
                template = fc.isel(time=0, drop=True)
                acc = xr.full_like(template, np.nan).astype("float32")
                ns = xr.zeros_like(template, dtype="int32")
            else:
                acc, ns = self._corr_map_time(fc_bin, obs_bin)

            acc = acc.expand_dims(lead_bin=[lab])
            ns = ns.expand_dims(lead_bin=[lab])
            acc_list.append(acc)
            ns_list.append(ns)

        acc_all = xr.concat(acc_list, dim="lead_bin")
        ns_all = xr.concat(ns_list, dim="lead_bin")
        return acc_all, ns_all

    # ----------------------------
    # NEW: member stats
    # ----------------------------
    def _compute_member_products(
        self,
        model_by_ens: Dict[str, List[str]],
        *,
        var: str,
        period: str,
        obs_da: xr.DataArray,
        lat_obs: str,
        lon_obs: str,
    ):
        """
        Returns a dict of member summary fields (+ optional full member maps).
        All outputs are lead_bin x lat x lon (except N_ENS_USED scalar).
        """
        acc_members = []
        ns_members = []
        ens_names = []

        for ens, files in model_by_ens.items():
            if not files:
                continue
            try:
                mem = self._open_var(files, var)
                mem = self._maybe_convert_geopotential_to_height(
                    mem, var_key=var, verbose=self.cfg.verbose
                )
                mem = self._maybe_convert_precip_to_mmday(
                    mem, var_key=var, verbose=self.cfg.verbose
                )
            except Exception:
                continue
            mem = self._select_period_time(mem, period)
            mem = self._to_daily_date_time(mem)

            lat_mod, lon_mod = self._guess_lat_lon_names(mem)
            mem = self._subset_region(mem, lat_mod, lon_mod, self.cfg.lat_range, self.cfg.lon_range)
            mem = self._ensure_latlon_names_match(
                mem, obs_da,
                mod_lat=lat_mod, mod_lon=lon_mod,
                obs_lat=lat_obs, obs_lon=lon_obs,
                tol=self.cfg.grid_tol,
            )

            mem_al, obs_al = self._align_and_check(mem, obs_da, lat_name=lat_obs, lon_name=lon_obs)

            mem_al = self._add_lead_day_coord(mem_al)
            obs_al = self._add_lead_day_coord(obs_al)

            acc_map, ns_map = self._compute_biweekly_maps(mem_al, obs_al)

            acc_members.append(acc_map)
            ns_members.append(ns_map)
            ens_names.append(ens)

        n_ens_used = len(acc_members)
        if n_ens_used == 0:
            # Create NaN placeholders based on obs grid
            template = obs_da.isel(time=0, drop=True)
            acc_nan = xr.full_like(template, np.nan).astype("float32")
            ns_zero = xr.zeros_like(template, dtype="int32")
            acc_nan = xr.concat([acc_nan.expand_dims(lead_bin=[lab]) for lab in self.cfg.lead_bin_labels], dim="lead_bin")
            ns_zero = xr.concat([ns_zero.expand_dims(lead_bin=[lab]) for lab in self.cfg.lead_bin_labels], dim="lead_bin")
            return {
                "ACC_MEMBER_MEAN": acc_nan,
                "ACC_MEMBER_STD": acc_nan,
                "P_ACC_POS": acc_nan,
                "N_SAMPLES_MEMBER_MEAN": ns_zero.astype("float32"),
                "N_ENS_USED": xr.DataArray(0, dims=()),
                "ACC_MEMBER": None,
                "N_SAMPLES_MEMBER": None,
            }

        acc_ens = xr.concat(acc_members, dim=xr.IndexVariable("ens", ens_names))
        ns_ens = xr.concat(ns_members, dim=xr.IndexVariable("ens", ens_names))

        acc_member_mean = acc_ens.mean("ens", skipna=True).astype("float32")
        acc_member_std = acc_ens.std("ens", skipna=True).astype("float32")
        p_acc_pos = (acc_ens > 0).mean("ens", skipna=True).astype("float32")
        ns_member_mean = ns_ens.mean("ens", skipna=True).astype("float32")

        out = {
            "ACC_MEMBER_MEAN": acc_member_mean,
            "ACC_MEMBER_STD": acc_member_std,
            "P_ACC_POS": p_acc_pos,
            "N_SAMPLES_MEMBER_MEAN": ns_member_mean,
            "N_ENS_USED": xr.DataArray(n_ens_used, dims=()),
            "ACC_MEMBER": acc_ens if self.cfg.save_member_maps else None,
            "N_SAMPLES_MEMBER": ns_ens if self.cfg.save_member_maps else None,
        }
        return out

    # ----------------------------
    # Main API (same signature style)
    # ----------------------------
    def compute(
        self,
        all_files: Dict[str, dict],
        *,
        period: str,
        obs_var_override: Optional[Dict[str, str]] = None,
    ) -> xr.Dataset:

        var_list = sorted(all_files.keys())
        if not var_list:
            raise ValueError("all_files is empty; nothing to compute.")

        first_var = var_list[0]
        exp_list = sorted(all_files[first_var].keys())
        if not exp_list:
            raise ValueError(f"No experiments found under var='{first_var}'.")

        acc_emean_out = []
        ns_emean_out = []

        # member outputs (optional)
        acc_mem_mean_out = []
        acc_mem_std_out = []
        p_pos_out = []
        n_ens_used_out = []
        ns_mem_mean_out = []
        acc_member_full_out = []
        ns_member_full_out = []

        for var in var_list:
            if not all_files[var]:
                continue

            any_exp = next(iter(all_files[var].keys()))
            obs_files = all_files[var][any_exp]["obs"]
            obs_var = (obs_var_override or {}).get(var, var)

            obs_da = self._open_var(obs_files, obs_var)
            # ---- unit fix for Z500: geopotential (m2/s2) -> geopotential height (m)
            obs_da = self._maybe_convert_geopotential_to_height(
                obs_da, var_key=var, verbose=self.cfg.verbose
            )
            obs_da = self._maybe_convert_precip_to_mmday(
                obs_da, var_key=var, verbose=self.cfg.verbose
            )
            
            obs_da = self._select_period_time(obs_da, period)
            obs_da = self._to_daily_date_time(obs_da)

            lat_obs, lon_obs = self._guess_lat_lon_names(obs_da)
            obs_da = self._subset_region(obs_da, lat_obs, lon_obs, self.cfg.lat_range, self.cfg.lon_range)

            if obs_da.sizes.get("time", 0) == 0:
                raise ValueError(f"OBS has no data after period selection for var={var} (obs_var={obs_var}).")

            acc_var = []
            ns_var = []

            # member stat containers per var
            acc_mem_mean_var = []
            acc_mem_std_var = []
            p_pos_var = []
            n_ens_used_var = []
            ns_mem_mean_var = []
            acc_member_full_var = []
            ns_member_full_var = []

            for exp in exp_list:
                if exp not in all_files[var]:
                    continue

                model_by_ens = all_files[var][exp]["model"]

                # ---- ensemble mean product (main)
                mod_mean = self._concat_ens_mean(model_by_ens, var=var, period=period)
                mod_mean = self._to_daily_date_time(mod_mean)
                mod_mean = self._maybe_convert_geopotential_to_height(
                        mod_mean, var_key=var, verbose=self.cfg.verbose
                )
                mod_mean = self._maybe_convert_precip_to_mmday(
                        mod_mean, var_key=var, verbose=self.cfg.verbose
                )
                
                lat_mod, lon_mod = self._guess_lat_lon_names(mod_mean)
                mod_mean = self._subset_region(mod_mean, lat_mod, lon_mod, self.cfg.lat_range, self.cfg.lon_range)
                mod_mean = self._ensure_latlon_names_match(
                    mod_mean, obs_da,
                    mod_lat=lat_mod, mod_lon=lon_mod,
                    obs_lat=lat_obs, obs_lon=lon_obs,
                    tol=self.cfg.grid_tol,
                )

                mod_al, obs_al = self._align_and_check(mod_mean, obs_da, lat_name=lat_obs, lon_name=lon_obs)
                mod_al = self._add_lead_day_coord(mod_al)
                obs_al = self._add_lead_day_coord(obs_al)

                acc_maps, ns_maps = self._compute_biweekly_maps(mod_al, obs_al)

                acc_maps = acc_maps.assign_coords(exp=exp).expand_dims("exp")
                ns_maps = ns_maps.assign_coords(exp=exp).expand_dims("exp")
                acc_var.append(acc_maps)
                ns_var.append(ns_maps)

                # ---- member stats (optional)
                if self.cfg.compute_member_stats:
                    mem_out = self._compute_member_products(
                        model_by_ens,
                        var=var,
                        period=period,
                        obs_da=obs_da,
                        lat_obs=lat_obs,
                        lon_obs=lon_obs,
                    )

                    acc_mem_mean_var.append(mem_out["ACC_MEMBER_MEAN"].assign_coords(exp=exp).expand_dims("exp"))
                    acc_mem_std_var.append(mem_out["ACC_MEMBER_STD"].assign_coords(exp=exp).expand_dims("exp"))
                    p_pos_var.append(mem_out["P_ACC_POS"].assign_coords(exp=exp).expand_dims("exp"))
                    ns_mem_mean_var.append(mem_out["N_SAMPLES_MEMBER_MEAN"].assign_coords(exp=exp).expand_dims("exp"))
                    n_ens_used_var.append(mem_out["N_ENS_USED"].assign_coords(exp=exp).expand_dims("exp"))

                    if self.cfg.save_member_maps and (mem_out["ACC_MEMBER"] is not None):
                        acc_member_full_var.append(mem_out["ACC_MEMBER"].assign_coords(exp=exp).expand_dims("exp"))
                        ns_member_full_var.append(mem_out["N_SAMPLES_MEMBER"].assign_coords(exp=exp).expand_dims("exp"))

            if not acc_var:
                continue

            acc_emean = xr.concat(acc_var, dim="exp").assign_coords(var=var).expand_dims("var")
            ns_emean = xr.concat(ns_var, dim="exp").assign_coords(var=var).expand_dims("var")
            acc_emean_out.append(acc_emean)
            ns_emean_out.append(ns_emean)

            if self.cfg.compute_member_stats:
                acc_mem_mean_out.append(xr.concat(acc_mem_mean_var, dim="exp").assign_coords(var=var).expand_dims("var"))
                acc_mem_std_out.append(xr.concat(acc_mem_std_var, dim="exp").assign_coords(var=var).expand_dims("var"))
                p_pos_out.append(xr.concat(p_pos_var, dim="exp").assign_coords(var=var).expand_dims("var"))
                ns_mem_mean_out.append(xr.concat(ns_mem_mean_var, dim="exp").assign_coords(var=var).expand_dims("var"))
                n_ens_used_out.append(xr.concat(n_ens_used_var, dim="exp").assign_coords(var=var).expand_dims("var"))

                if self.cfg.save_member_maps and acc_member_full_var:
                    acc_member_full_out.append(xr.concat(acc_member_full_var, dim="exp").assign_coords(var=var).expand_dims("var"))
                    ns_member_full_out.append(xr.concat(ns_member_full_var, dim="exp").assign_coords(var=var).expand_dims("var"))

        if not acc_emean_out:
            raise ValueError("No ACC map results produced (check inputs and period).")

        ACC_EMEAN = xr.concat(acc_emean_out, dim="var").astype("float32")
        N_SAMPLES_EMEAN = xr.concat(ns_emean_out, dim="var").astype("int32")

        data_vars = {
            "ACC_EMEAN": ACC_EMEAN,
            "N_SAMPLES_EMEAN": N_SAMPLES_EMEAN,
        }

        if self.cfg.compute_member_stats:
            data_vars.update({
                "ACC_MEMBER_MEAN": xr.concat(acc_mem_mean_out, dim="var").astype("float32"),
                "ACC_MEMBER_STD": xr.concat(acc_mem_std_out, dim="var").astype("float32"),
                "P_ACC_POS": xr.concat(p_pos_out, dim="var").astype("float32"),
                "N_SAMPLES_MEMBER_MEAN": xr.concat(ns_mem_mean_out, dim="var").astype("float32"),
                "N_ENS_USED": xr.concat(n_ens_used_out, dim="var").astype("int32"),
            })

        if self.cfg.save_member_maps and acc_member_full_out:
            data_vars["ACC_MEMBER"] = xr.concat(acc_member_full_out, dim="var").astype("float32")
            data_vars["N_SAMPLES_MEMBER"] = xr.concat(ns_member_full_out, dim="var").astype("int32")

        ds = xr.Dataset(
            data_vars=data_vars,
            coords=dict(
                var=ACC_EMEAN["var"],
                exp=ACC_EMEAN["exp"],
                lead_bin=ACC_EMEAN["lead_bin"],
            ),
            attrs=dict(
                period=period,
                init_time=self.cfg.init_time,
                lead_bins=",".join([f"{a}-{b}" for a, b in self.cfg.lead_bins]),
                lead_bin_labels=",".join(self.cfg.lead_bin_labels),
                lat_range=str(self.cfg.lat_range),
                lon_range=str(self.cfg.lon_range),
                require_same_grid=str(self.cfg.require_same_grid),
                grid_tol=str(self.cfg.grid_tol),
                min_samples=str(self.cfg.min_samples),
                compute_member_stats=str(self.cfg.compute_member_stats),
                save_member_maps=str(self.cfg.save_member_maps),
                note="ACC_* are correlation across TIME at each grid point within each lead-bin. "
                     "EMEAN uses ensemble-mean first. MEMBER_* summarizes per-member ACC maps.",
            ),
        )
        return ds

    def save(self, ds: xr.Dataset, out_nc: Optional[str] = None) -> str:
        path = out_nc or self.cfg.out_nc
        if (not self.cfg.overwrite) and os.path.exists(path):
            raise FileExistsError(f"Output exists: {path}")
        ds.to_netcdf(path)
        return path

    def save_per_var_exp(
        self,
        ds: xr.Dataset,
        *,
        out_dir: str,
        group: str,
        freq: str,
        run: str,
        obs: str,
        overwrite: Optional[bool] = None,
    ) -> List[str]:
        """
        Split ds into per (var, exp) netcdf files:
          s2s_accmap_{group}_{freq}_{run}_{obs}_{exp}_{var}.nc
        """
        out = []
        ow = self.cfg.overwrite if overwrite is None else overwrite
        os.makedirs(out_dir, exist_ok=True)

        for var in ds["var"].values.tolist():
            for exp in ds["exp"].values.tolist():
                ds_one = ds.sel(var=var, exp=exp)
                fn = f"s2s_accmap_{group}_{freq}_{run}_{obs}_{exp}_{var}.nc"
                path = os.path.join(out_dir, fn)
                if (not ow) and os.path.exists(path):
                    out.append(path)
                    continue
                ds_one.to_netcdf(path)
                out.append(path)
        return out

In [15]:
if __name__ == "__main__":
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/acc_rmse"
    land_mask = "/compyfs/zhan391/acme_init/lnd_sea_mask/landmask_1x1.nc"
    os.makedirs(out_path, exist_ok=True)

    exp_list = {
        "Jan2012": dict(models=["CTRL10-S0","CAPT10-S0","DART20-S0","DART40-S0"],
                        period="201201-201203", season="Winter", init_month=1),
        "Jun2012": dict(models=["CTRL10-S1","CAPT10-S1","DART40-S1"],
                        period="201206-201208", season="Summer", init_month=6),
    }

    s2s_var_dict = {
        "UBOT": "U10",
        "VBOT": "V10",
        "PSL": "PSL",
        "PRECT": "PRECT",
        "U850": "U850",
        "V850": "V850",
        "T850": "T850",
        "Q850": "Q850",
        "Z500": "Z500",
        "OMEGA500": "OMEGA500",
        "U200": "U200",
        "V200": "V200",
    }
    
    s2s_var_dict = {
        "TREFHT": "TREFHT",
    }
    
    group = "Jan2012"
    freq = "daily"
    run = "fc"
    obs = "ERA5"            # <-- this is the obs_name you should pass
    obs_name = obs          # <-- make it explicit to avoid confusion
    lat_range = (-90, 90)
    lon_range = None

    ens_start = 1
    ens_prefix = "EN"
    ens_width = 2
    overwrite = False

    init_time = "2012-01-01"
    lead_bins = ((1,7), (7, 14), (15, 28), (29, 42), (43, 56))
    lead_bin_labels = ("wk1", "wk2", "wk3-4", "wk5-6", "wk7-8")
    min_samples = 10
    compute_member_stats=True
    save_member_maps=True   # keep False unless you really want ACC_MEMBER

    exp_dict = build_experiments(data_path)
    period = exp_list[group]["period"]
    models_for_group = exp_list[group]["models"]   # <-- IMPORTANT

    model_template = "%(var)s.%(ens)s.%(year)d.nc"

    cfg_collect = FileCollectionConfig(
        group=group,
        freq=freq,
        run=run,
        obs=obs_name,
        period=period,
        model_template=model_template,
        ens_prefix=ens_prefix,
        ens_width=ens_width,
        ens_start=ens_start,
    )

    collector = S2SFileCollector(
        exp_list=exp_list,
        exp_dict=exp_dict,
        obs_registry=OBS_REGISTRY,
        s2s_var_dict=s2s_var_dict,
        get_obs_file_func=get_obs_file,
    )

    # 1) collect files
    all_files = collector.collect(cfg_collect, verbose=True)
    print(f"Collected variables: {list(all_files.keys())}")

    # 2) ACC-map assessor (biweekly maps)
    acc_cfg = BiweeklyAccMapConfig(
        init_time=init_time,
        lead_bins=lead_bins,
        lead_bin_labels=lead_bin_labels,
        min_samples=min_samples,
        lat_range=lat_range,
        lon_range=lon_range,
        overwrite=overwrite,
        compute_member_stats=compute_member_stats,
        save_member_maps=save_member_maps,   # keep False unless you really want ACC_MEMBER
    )
    acc_map_assessor = BiweeklyAccMapAssessor(acc_cfg)

    # 3) loop vars, compute+save per-var
    for var, obs_var in s2s_var_dict.items():
        if var not in all_files:
            continue
    
        # get the collector output for this var (dict keyed by exp)
        one_var_files = {var: all_files[var]}   # must be a dict, no trailing comma
    
        # Biweekly ACC maps (loops models internally)
        ds_accmap = acc_map_assessor.compute(
            one_var_files,
            period=period,
            obs_var_override=s2s_var_dict,   # UBOT->U10, etc
        )
        
        print("ACC vars:", list(ds_accmap.data_vars))
        print("ACC_EMEAN finite frac:",
              float(np.isfinite(ds_accmap["ACC_EMEAN"]).mean()))
        print("N_SAMPLES_EMEAN min/max:",
              int(ds_accmap["N_SAMPLES_EMEAN"].min()),
              int(ds_accmap["N_SAMPLES_EMEAN"].max()))
        
        # One file per (var,exp)
        saved = acc_map_assessor.save_per_var_exp(
            ds_accmap,
            out_dir=out_path,
            group=group,
            freq=freq,
            run=run,
            obs=obs,
        )
        print(f"[{var}] Saved {len(saved)} ACC-map files")
        

Collected variables: ['TREFHT']


/tmp/ipykernel_45597/876256618.py:946: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'ens' ('ens',) The recommendation is to set join explicitly for this case.
  acc_member_full_out.append(xr.concat(acc_member_full_var, dim="exp").assign_coords(var=var).expand_dims("var"))
/tmp/ipykernel_45597/876256618.py:947: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'ens' ('ens',) The recommendation is to set join explicitly for this case.
  ns_member_full_out.append(xr.concat(ns_member_full_var, dim="exp").assign_coords(v

ACC vars: ['ACC_EMEAN', 'N_SAMPLES_EMEAN', 'ACC_MEMBER_MEAN', 'ACC_MEMBER_STD', 'P_ACC_POS', 'N_SAMPLES_MEMBER_MEAN', 'N_ENS_USED', 'ACC_MEMBER', 'N_SAMPLES_MEMBER']
ACC_EMEAN finite frac: 0.6
N_SAMPLES_EMEAN min/max: 7 14
[TREFHT] Saved 4 ACC-map files
